In [31]:
import import_ipynb
from rco_test import *
from ast import *
from utils import *
from x86_ast import *


In [32]:
def select_arg(e: expr) -> arg:
        # YOUR CODE HERE
        match e:
            case Constant(value):
                return Immediate(value)
            case Name(var):
                return Variable(var)
         

In [33]:
def select_stmt(s: stmt) -> list[instr]:
        # YOUR CODE HERE
        match s:
          
            case Assign([Name(var)],UnaryOp(USub(), v)):
                val = select_arg(v)
                return [Instr('movq',[val, Reg('rax')]),
                        Instr('negq',[Reg('rax')]),
                        Instr('movq',[Reg('rax'), Variable(var)])]
            case Assign([Name(var)], BinOp(left,Add(), right)):
                l = select_arg(left)
                r = select_arg(right)
                if isinstance(left, Name) and left.id == var:
                     return [Instr('addq',[r,Variable(var)])]
                elif isinstance(right, Name) and right.id == var:
                     return [Instr('addq',[l,Variable(var)])]
                else:
                    return [Instr('movq',[l, Reg('rax')]),
                        Instr('addq',[r,Reg('rax')]),
                        Instr('movq',[Reg('rax'), Variable(var)])]
            case Assign([Name(var)], BinOp(left, Sub(), right)):
                l = select_arg(left)
                r = select_arg(right)
                if isinstance(left, Name) and left.id == var:
                     return [Instr('subq',[r,Variable(var)])]
                elif isinstance(right, Name) and right.id == var:
                     return [Instr('subq', [l, Variable(var)])]
                else:
                     return [Instr('movq',[l,Reg('rax')]),
                        Instr('subq',[r,Reg('rax')]),
                        Instr('movq', [Reg('rax'),Variable(var)])]
            case Assign([Name(var)], Call(Name('input_int'))):
                return [Callq('read_int',1),
                        Instr('movq', [Reg('rax'), Variable(var)])]
            case Assign([Name(var)],value):
                  new_value = select_arg(value)
                  return [Instr('movq',[new_value,Variable(var)])]
            case Expr(Call(Name('print'),[arg])):
                new_arg = select_arg(arg)
                return [Instr('movq',[new_arg, Reg('rdi')]), 
                        Callq('print_int',1)]
           

In [ ]:
def select_instruction(p : Module) -> X86Program:
    match p:
        case Module(body):
            new_body = []
            for stmt in body:
                new_body.extend(select_stmt(stmt))
                
            return X86Program(new_body)

In [35]:
test = """
x = 8 + -10
x = x + 2
print(x)
"""

In [36]:
parsed_test = parse(test)

In [37]:
print(dump(parsed_test,indent=4))

Module(
    body=[
        Assign(
            targets=[
                Name(id='x', ctx=Store())],
            value=BinOp(
                left=Constant(value=8),
                op=Add(),
                right=UnaryOp(
                    op=USub(),
                    operand=Constant(value=10)))),
        Assign(
            targets=[
                Name(id='x', ctx=Store())],
            value=BinOp(
                left=Name(id='x', ctx=Load()),
                op=Add(),
                right=Constant(value=2))),
        Expr(
            value=Call(
                func=Name(id='print', ctx=Load()),
                args=[
                    Name(id='x', ctx=Load())]))])


In [38]:
rco_code = remove_complex_operands(parsed_test)

In [39]:
print(dump(rco_code,indent=4))

Module(
    body=[
        Assign(
            targets=[
                Name(id='temp.5', ctx=Load())],
            value=UnaryOp(
                op=USub(),
                operand=Constant(value=10))),
        Assign(
            targets=[
                Name(id='x', ctx=Load())],
            value=BinOp(
                left=Constant(value=8),
                op=Add(),
                right=Name(id='temp.5', ctx=Load()))),
        Assign(
            targets=[
                Name(id='x', ctx=Load())],
            value=BinOp(
                left=Name(id='x', ctx=Load()),
                op=Add(),
                right=Constant(value=2))),
        Expr(
            value=Call(
                func=Name(id='print', ctx=Load()),
                args=[
                    Name(id='x', ctx=Load())]))])


In [40]:
print(select_instruction(rco_code))

	.globl main
main:
    movq $10, %rax
    negq %rax
    movq %rax, temp.5
    movq $8, %rax
    addq temp.5, %rax
    movq %rax, x
    addq $2, x
    movq x, %rdi
    callq print_int


